In [1]:
import pandas as pd
from IPython.display import display
import yfinance as yf

import os

In [4]:
def load_process_data(file, hr=2): #hr用于调整时区
    df = pd.read_csv(file)
    df = df[["date","open","high","low","close","volume"]]
    df = df.reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])

    df.columns = ["Datetime","Open","High","Low","Close","Volume"]
    df = df.set_index("Datetime")
    df = df.sort_index()
    df.index = df.index + pd.Timedelta(hours=hr) #调整时区
    df = df.between_time("09:30", "15:59")
    df = df[~df.index.duplicated(keep="first")]
    
    return df 

def load_process_yf_data():
    df = yf.download(tickers="^GSPC", period="8d", interval="1m")
    df.columns = df.columns.droplevel(1)
    df = df.reset_index()
    df = df[["Datetime","Close","High","Low","Open","Volume"]]
    df = df.rename(columns = {"Price":"Index"})
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.set_index("Datetime")
    df = df[~df.index.duplicated(keep="first")]
    df = df.sort_index()

    df.index = df.index + pd.Timedelta(hours=-5)

    return df

def load_process_vix():
    df_vix = pd.read_csv("raw_data/vix_daily.csv")
    df_vix["Datetime"] = pd.to_datetime(df_vix["date"])
    df_vix = df_vix.set_index("Datetime")
    df_vix = df_vix.drop(columns=["date"])
    df_vix.columns = ["Open","High","Low","Close"]
    df_vix.to_csv("processed_data/prc_vix_daily_1990-2024.csv")
    return df_vix

In [5]:
load_process_vix()

,Open,High,Low,Close
Datetime,,,,
1990-01-02,17.240000,17.240000,17.240000,17.240000
1990-01-03,18.190001,18.190001,18.190001,18.190001
1990-01-04,19.219999,19.219999,19.219999,19.219999
1990-01-05,20.110001,20.110001,20.110001,20.110001
1990-01-08,20.260000,20.260000,20.260000,20.260000
...,...,...,...,...
2024-06-17,13.070000,13.290000,12.500000,12.750000
2024-06-18,12.700000,12.740000,12.240000,12.300000
2024-06-20,12.500000,13.550000,12.180000,13.280000


处理跨资产数据

In [17]:
# 识别raw_data文件夹下的所有以multimkt开头的csv文件
files = [f for f in os.listdir("raw_data") if f.startswith("multimkt") and f.endswith(".csv")]
dfs = []
for file in files:
    df = pd.read_csv(os.path.join("raw_data", file))
    dfs.append(df)

df_multimkt = pd.concat(dfs, ignore_index=True)
sym = df_multimkt["sym_root"].unique()
sym
df_multimkt.head(20)
df_multimkt.count()

sym_root       3261508
date           3261508
minute         3261508
open_price     3261508
high_price     3261508
low_price      3261508
close_price    3261508
volume         3261508
amount         3261508
dtype: int64

In [18]:
for i in sym:
    df = df_multimkt[df_multimkt["sym_root"] == i].copy()
    df["Datetime"] = pd.to_datetime(df["date"] + " " + df["minute"])
    df.drop(columns=["date","minute","sym_root","amount"], inplace=True)
    df.rename(columns={"open_price":"Open","high_price":"High","low_price":"Low","close_price":"Close","volume":"Volume"}, inplace=True)
    df = df.sort_values("Datetime")
    df = df.reset_index(drop=True)
    display(df.head())
    #to_csv
    df.to_csv(f"processed_data/prc_1_min_{i}_2012-2021.csv", index=False, encoding="utf-8-sig")

,Open,High,Low,Close,Volume,Datetime
0,117.94,117.9899,117.91,117.9899,224074,2014-01-02 09:30:00
1,117.99,118.0300,117.98,118.0100,3786,2014-01-02 09:31:00
2,117.98,118.0000,117.96,117.9799,1261,2014-01-02 09:32:00
3,117.98,117.9800,117.96,117.9800,1629,2014-01-02 09:33:00
4,117.99,117.9900,117.94,117.9600,1676,2014-01-02 09:34:00


,Open,High,Low,Close,Volume,Datetime
0,101.7100,101.72,101.70,101.72,11988,2014-01-02 09:30:00
1,101.7499,101.76,101.70,101.70,511,2014-01-02 09:31:00
2,101.7300,101.73,101.71,101.72,205,2014-01-02 09:32:00
3,101.7100,101.71,101.70,101.70,290,2014-01-02 09:33:00
4,101.7200,101.73,101.72,101.73,362,2014-01-02 09:34:00


,Open,High,Low,Close,Volume,Datetime
0,34.70,34.70,34.6401,34.67,40110,2014-01-02 09:30:00
1,34.66,34.66,34.6400,34.64,166,2014-01-02 09:31:00
2,34.64,34.64,34.6300,34.64,165,2014-01-02 09:32:00
3,34.63,34.63,34.6300,34.63,36,2014-01-02 09:33:00
4,34.63,34.63,34.6200,34.62,149,2014-01-02 09:34:00


,Open,High,Low,Close,Volume,Datetime
0,21.670,21.670,21.670,21.670,13488,2014-01-02 09:30:00
1,21.665,21.665,21.665,21.665,200,2014-01-02 09:31:00
2,21.660,21.660,21.660,21.660,100,2014-01-02 09:32:00
3,21.660,21.660,21.660,21.660,25,2014-01-02 09:41:00
4,21.655,21.655,21.655,21.655,69,2014-01-02 09:42:00


,Open,High,Low,Close,Volume,Datetime
0,64.2900,64.4900,64.2600,64.40,515715,2014-01-02 09:30:00
1,64.3600,64.3999,64.3200,64.39,446,2014-01-02 09:31:00
2,64.4000,64.4900,64.3801,64.49,1229,2014-01-02 09:32:00
3,64.4600,64.4700,64.4400,64.44,330,2014-01-02 09:33:00
4,64.4401,64.4600,64.4300,64.46,398,2014-01-02 09:34:00


处理跨市场数据（恒生）

In [19]:
for i in sym:
    df = df_multimkt[df_multimkt["sym_root"] == i].copy()
    df["Datetime"] = pd.to_datetime(df["date"] + " " + df["minute"])
    df.drop(columns=["date","minute","sym_root","amount"], inplace=True)
    df.rename(columns={"open_price":"Open","high_price":"High","low_price":"Low","close_price":"Close","volume":"Volume"}, inplace=True)
    df = df.sort_values("Datetime")
    df = df.reset_index(drop=True)
    display(df.head())
    #to_csv
    df.to_csv(f"processed_data/prc_1_min_{i}_2012-2021.csv", index=False, encoding="utf-8-sig")

,Open,High,Low,Close,Volume,Datetime
0,117.94,117.9899,117.91,117.9899,224074,2014-01-02 09:30:00
1,117.99,118.0300,117.98,118.0100,3786,2014-01-02 09:31:00
2,117.98,118.0000,117.96,117.9799,1261,2014-01-02 09:32:00
3,117.98,117.9800,117.96,117.9800,1629,2014-01-02 09:33:00
4,117.99,117.9900,117.94,117.9600,1676,2014-01-02 09:34:00


,Open,High,Low,Close,Volume,Datetime
0,101.7100,101.72,101.70,101.72,11988,2014-01-02 09:30:00
1,101.7499,101.76,101.70,101.70,511,2014-01-02 09:31:00
2,101.7300,101.73,101.71,101.72,205,2014-01-02 09:32:00
3,101.7100,101.71,101.70,101.70,290,2014-01-02 09:33:00
4,101.7200,101.73,101.72,101.73,362,2014-01-02 09:34:00


,Open,High,Low,Close,Volume,Datetime
0,34.70,34.70,34.6401,34.67,40110,2014-01-02 09:30:00
1,34.66,34.66,34.6400,34.64,166,2014-01-02 09:31:00
2,34.64,34.64,34.6300,34.64,165,2014-01-02 09:32:00
3,34.63,34.63,34.6300,34.63,36,2014-01-02 09:33:00
4,34.63,34.63,34.6200,34.62,149,2014-01-02 09:34:00


,Open,High,Low,Close,Volume,Datetime
0,21.670,21.670,21.670,21.670,13488,2014-01-02 09:30:00
1,21.665,21.665,21.665,21.665,200,2014-01-02 09:31:00
2,21.660,21.660,21.660,21.660,100,2014-01-02 09:32:00
3,21.660,21.660,21.660,21.660,25,2014-01-02 09:41:00
4,21.655,21.655,21.655,21.655,69,2014-01-02 09:42:00


,Open,High,Low,Close,Volume,Datetime
0,64.2900,64.4900,64.2600,64.40,515715,2014-01-02 09:30:00
1,64.3600,64.3999,64.3200,64.39,446,2014-01-02 09:31:00
2,64.4000,64.4900,64.3801,64.49,1229,2014-01-02 09:32:00
3,64.4600,64.4700,64.4400,64.44,330,2014-01-02 09:33:00
4,64.4401,64.4600,64.4300,64.46,398,2014-01-02 09:34:00


In [20]:
df.head(20)

,Open,High,Low,Close,Volume,Datetime
0,64.2900,64.4900,64.2600,64.4000,515715,2014-01-02 09:30:00
1,64.3600,64.3999,64.3200,64.3900,446,2014-01-02 09:31:00
2,64.4000,64.4900,64.3801,64.4900,1229,2014-01-02 09:32:00
3,64.4600,64.4700,64.4400,64.4400,330,2014-01-02 09:33:00
4,64.4401,64.4600,64.4300,64.4600,398,2014-01-02 09:34:00
5,64.4800,64.5200,64.4500,64.4600,440,2014-01-02 09:35:00
6,64.4300,64.4300,64.3200,64.4000,434,2014-01-02 09:36:00
7,64.4200,64.4700,64.3880,64.4290,332,2014-01-02 09:37:00
8,64.4500,64.4700,64.3600,64.3700,435,2014-01-02 09:38:00
9,64.3900,64.3900,64.2801,64.3299,640,2014-01-02 09:39:00


In [21]:
# 识别raw_data文件夹下的所有以RESSET开头的xlsx文件
files = [f for f in os.listdir("raw_data") if f.startswith("RESSET") and f.endswith(".xlsx")]
dfs = []
for file in files:
    df = pd.read_excel(os.path.join("raw_data", file))
    if '2019' in file or '2020' in file or '2021' in file:
        df.columns = df.columns.str.split('_', n=1).str[-1]
    display(df.head())
    dfs.append(df)

df_concat = pd.concat(dfs, ignore_index=True)

,Code_Mkt,Secunm,MTime,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1
0,159920.SZ,恒生ETF,13:06:07,2016-01-08 13:05:59,2016-01-08,13:06,1.058,1.057,1.057,34000,35961.0,5
1,159920.SZ,恒生ETF,13:06:58,2016-01-08 13:06:53,2016-01-08,13:07,1.057,1.057,1.057,9400,9935.8,1
2,159920.SZ,恒生ETF,13:08:06,2016-01-08 13:07:59,2016-01-08,13:08,1.057,1.056,1.057,285500,301594.5,17
3,159920.SZ,恒生ETF,13:09:04,2016-01-08 13:08:56,2016-01-08,13:09,1.056,1.056,1.056,0,0.0,0
4,159920.SZ,恒生ETF,13:10:06,2016-01-08 13:09:59,2016-01-08,13:10,1.057,1.056,1.057,152300,160954.0,12


,Code_Mkt,Secunm,MTime,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1
0,159920.SZ,恒生ETF,09:15:11,2017-01-03 09:15:09,2017-01-03,09:16,NaN,NaN,NaN,0,0.0,0
1,159920.SZ,恒生ETF,09:17:51,2017-01-03 09:17:51,2017-01-03,09:18,NaN,NaN,NaN,0,0.0,0
2,159920.SZ,恒生ETF,09:19:58,2017-01-03 09:19:57,2017-01-03,09:20,NaN,NaN,NaN,0,0.0,0
3,159920.SZ,恒生ETF,09:20:09,2017-01-03 09:20:09,2017-01-03,09:21,NaN,NaN,NaN,0,0.0,0
4,159920.SZ,恒生ETF,09:24:57,2017-01-03 09:24:57,2017-01-03,09:25,NaN,NaN,NaN,0,0.0,0


,Code_Mkt,Secunm,MTime,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1
0,159920.SZ,恒生ETF,09:15:11,2018-01-02 09:15:12,2018-01-02,09:16,NaN,NaN,NaN,0,0.0,0
1,159920.SZ,恒生ETF,09:17:17,2018-01-02 09:17:18,2018-01-02,09:18,NaN,NaN,NaN,0,0.0,0
2,159920.SZ,恒生ETF,09:21:05,2018-01-02 09:21:06,2018-01-02,09:22,NaN,NaN,NaN,0,0.0,0
3,159920.SZ,恒生ETF,09:22:43,2018-01-02 09:22:45,2018-01-02,09:23,NaN,NaN,NaN,0,0.0,0
4,159920.SZ,恒生ETF,09:23:28,2018-01-02 09:23:30,2018-01-02,09:24,NaN,NaN,NaN,0,0.0,0


,Code_Mkt,Secunm,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1,Amplitude_Min,AvgPr_Min,Ret_Min
0,159920.SZ,恒生ETF,2019-02-19 14:28:57,2019-02-19,14:29,1.484,1.484,1.484,60300,89485.20,1,0.0000,1.4840,0.000000
1,159920.SZ,恒生ETF,2019-02-19 14:29:57,2019-02-19,14:30,1.485,1.484,1.484,16300,24196.60,2,0.0672,1.4845,0.000000
2,159920.SZ,恒生ETF,2019-02-19 14:30:57,2019-02-19,14:31,1.485,1.484,1.484,2669500,3964198.50,51,0.0672,1.4850,0.000674
3,159920.SZ,恒生ETF,2019-02-19 14:31:57,2019-02-19,14:32,1.485,1.485,1.485,561571,833932.94,4,0.0000,1.4850,0.000000
4,159920.SZ,恒生ETF,2019-02-19 14:32:57,2019-02-19,14:33,1.486,1.485,1.486,596200,885363.70,4,0.0672,1.4850,0.000000


,Code_Mkt,Secunm,MTime,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1
0,159920.SZ,恒生ETF,11:21:03,2012-10-22 11:20:58,2012-10-22,11:21,0.997,0.997,0.997,841078,838554.77,18
1,159920.SZ,恒生ETF,11:22:03,2012-10-22 11:21:58,2012-10-22,11:22,0.997,0.997,0.997,281009,280165.97,8
2,159920.SZ,恒生ETF,11:23:10,2012-10-22 11:22:58,2012-10-22,11:23,0.997,0.997,0.997,573031,571311.91,15
3,159920.SZ,恒生ETF,11:24:06,2012-10-22 11:23:58,2012-10-22,11:24,0.997,0.997,0.997,833236,830736.29,20
4,159920.SZ,恒生ETF,11:24:58,2012-10-22 11:24:55,2012-10-22,11:25,0.997,0.997,0.997,353523,352462.43,14


,Code_Mkt,Secunm,MTime,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1
0,159920.SZ,恒生ETF,11:02:40,2013-01-10 11:02:46,2013-01-10,11:03,1.038,1.038,1.038,0,0.00,0
1,159920.SZ,恒生ETF,14:11:58,2013-01-04 14:11:58,2013-01-04,14:12,1.025,1.025,1.025,27169,27848.22,1
2,159920.SZ,恒生ETF,10:37:30,2013-01-25 10:37:55,2013-01-25,10:38,1.055,1.055,1.055,0,0.00,0
3,159920.SZ,恒生ETF,14:12:58,2013-01-04 14:12:52,2013-01-04,14:13,1.025,1.025,1.025,0,0.00,0
4,159920.SZ,恒生ETF,11:03:47,2013-01-10 11:03:55,2013-01-10,11:04,1.040,1.038,1.038,105100,109098.40,3


,Code_Mkt,Secunm,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1,Amplitude_Min,AvgPr_Min,Ret_Min
0,159920.SZ,恒生ETF,2021-01-13 15:25:00,2021-01-13,15:26,1.47,1.47,1.47,0,0.0,0,0.0,NaN,0.0
1,159920.SZ,恒生ETF,2021-01-13 15:26:00,2021-01-13,15:27,1.47,1.47,1.47,0,0.0,0,0.0,NaN,0.0
2,159920.SZ,恒生ETF,2021-01-13 15:27:00,2021-01-13,15:28,1.47,1.47,1.47,0,0.0,0,0.0,NaN,0.0
3,159920.SZ,恒生ETF,2021-01-13 15:28:00,2021-01-13,15:29,1.47,1.47,1.47,0,0.0,0,0.0,NaN,0.0
4,159920.SZ,恒生ETF,2021-01-13 15:29:00,2021-01-13,15:30,1.47,1.47,1.47,0,0.0,0,0.0,NaN,0.0


,Code_Mkt,Secunm,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1,Amplitude_Min,AvgPr_Min,Ret_Min
0,159920.SZ,恒生ETF,2020-01-02 09:25:42,2020-01-02,09:26,1.555,1.555,1.555,1008500,1568217.5,52.0,0.0,1.555,NaN
1,159920.SZ,恒生ETF,2020-01-02 09:26:42,2020-01-02,09:27,1.555,1.555,1.555,0,0.0,0.0,0.0,NaN,0.0
2,159920.SZ,恒生ETF,2020-01-02 09:27:42,2020-01-02,09:28,1.555,1.555,1.555,0,0.0,0.0,0.0,NaN,0.0
3,159920.SZ,恒生ETF,2020-01-02 09:28:42,2020-01-02,09:29,1.555,1.555,1.555,0,0.0,0.0,0.0,NaN,0.0
4,159920.SZ,恒生ETF,2020-01-02 09:29:12,2020-01-02,09:30,1.555,1.555,1.555,0,0.0,0.0,0.0,NaN,0.0


,Code_Mkt,Secunm,MTime,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1
0,159920.SZ,恒生ETF,14:38:10,2015-02-25 13:34:51,2015-02-25,13:35,1.154,1.154,1.154,0,0.0,0
1,159920.SZ,恒生ETF,14:40:33,2015-02-25 13:35:54,2015-02-25,13:36,1.155,1.153,1.154,119900,138346.1,5
2,159920.SZ,恒生ETF,14:42:52,2015-02-25 13:36:57,2015-02-25,13:37,1.153,1.153,1.153,0,0.0,0
3,159920.SZ,恒生ETF,14:44:39,2015-02-25 13:37:45,2015-02-25,13:38,1.153,1.153,1.153,0,0.0,0
4,159920.SZ,恒生ETF,14:45:52,2015-02-25 13:38:34,2015-02-25,13:39,1.155,1.155,1.155,17900,20674.5,2


,Code_Mkt,Secunm,MTime,QTime,Qdate,StdTime,Highpr,Lowpr,Begpr,TVolume_accu1,TSum_accu1,TDeals_accu1
0,159920.SZ,恒生ETF,14:23:01,2014-01-14 13:46:50,2014-01-14,13:47,1.009,1.009,1.009,100,100.9,1
1,159920.SZ,恒生ETF,14:24:49,2014-01-14 13:47:50,2014-01-14,13:48,1.009,1.009,1.009,0,0.0,0
2,159920.SZ,恒生ETF,14:26:13,2014-01-14 13:48:35,2014-01-14,13:49,1.009,1.009,1.009,0,0.0,0
3,159920.SZ,恒生ETF,14:28:29,2014-01-14 13:49:50,2014-01-14,13:50,1.009,1.009,1.009,0,0.0,0
4,159920.SZ,恒生ETF,14:29:51,2014-01-14 13:50:35,2014-01-14,13:51,1.009,1.009,1.009,0,0.0,0


In [22]:
# 导出 csv
df = df_concat[['Qdate','StdTime','Highpr','Lowpr','Begpr','TVolume_accu1']].copy()

df["Datetime"] = pd.to_datetime(
    df["Qdate"].astype(str) + " " + df["StdTime"].astype(str)
)

df.drop(columns=["Qdate","StdTime"], inplace=True)

df.rename(columns={
    "Begpr":"Open",
    "Highpr":"High",
    "Lowpr":"Low",
    "TVolume_accu1":"Volume"
}, inplace=True)

df = df.set_index("Datetime")
df = df.sort_index()

# 去掉开盘/收盘集合竞价影响，只保留连续竞价时段
df = df.between_time("09:31", "15:59")

# 时间前移一分钟
df.index = df.index - pd.Timedelta(minutes=1)

# 构造 Close
df = df.groupby(df.index.date, group_keys=False).apply(
    lambda x: x.assign(Close=x["Open"].shift(-1).fillna(x["Open"].iat[-1]))
)

df = df.reset_index()

display(df.head())

df.to_csv("processed_data/prc_1_min_HK_2012-2021.csv", index=False, encoding="utf-8-sig")

,Datetime,High,Low,Open,Volume,Close
0,2012-10-22 09:30:00,0.984,0.979,0.981,13806232,0.982
1,2012-10-22 09:31:00,0.987,0.982,0.982,5331238,0.985
2,2012-10-22 09:32:00,0.986,0.984,0.985,5090390,0.985
3,2012-10-22 09:33:00,0.987,0.985,0.985,5255962,0.987
4,2012-10-22 09:34:00,0.989,0.987,0.987,6691753,0.987


In [23]:
display(df.tail())

,Datetime,High,Low,Open,Volume,Close
576866,2021-12-31 15:26:00,1.212,1.212,1.212,0,1.212
576867,2021-12-31 15:27:00,1.212,1.212,1.212,0,1.212
576868,2021-12-31 15:28:00,1.212,1.212,1.212,0,1.212
576869,2021-12-31 15:29:00,1.212,1.212,1.212,0,1.212
576870,2021-12-31 15:30:00,1.212,1.212,1.212,0,1.212
